## 0.提取文章

In [37]:
import os
import json
import xml.etree.ElementTree as ET

# 路径：notebook在 notebooks/ 下运行，所以用 ../ 回到项目根目录
xml_dir = "../data/raw/PMC000xxxxxx"
output_path = "../data/processed/dataset.json"

files = [f for f in os.listdir(xml_dir) if f.endswith(".xml")]
print(f"共发现 {len(files)} 个XML文件")

records = []

for fname in files:
    path = os.path.join(xml_dir, fname)
    try:
        tree = ET.parse(path)
        root = tree.getroot()

        title_el = root.find(".//article-title")
        title = "".join(title_el.itertext()).strip() if title_el is not None else None

        abstract_el = root.find(".//abstract")
        abstract = " ".join(abstract_el.itertext()).strip() if abstract_el is not None else None

        journal_el = root.find(".//journal-meta//journal-title")
        journal = journal_el.text.strip() if journal_el is not None and journal_el.text else None

        pmid_el = root.find('.//article-id[@pub-id-type="pmid"]')
        pmid = pmid_el.text.strip() if pmid_el is not None and pmid_el.text else None

        pub_date = None
        for pub_type in ["ppub", "epub", "pmc-release"]:
            date_el = root.find(f'.//pub-date[@pub-type="{pub_type}"]')
            if date_el is not None:
                year = date_el.findtext("year")
                month = date_el.findtext("month")
                day = date_el.findtext("day")
                if year:
                    pub_date = "-".join(filter(None, [year, month, day]))
                    break

        records.append({
            "pmc_id": fname.replace(".xml", ""),
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "journal": journal,
            "pub_date": pub_date,
        })
    except Exception as e:
        records.append({
            "pmc_id": fname.replace(".xml", ""),
            "pmid": None, "title": None, "abstract": None,
            "journal": None, "pub_date": None,
            "_parse_error": str(e),
        })

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=1)

print(f"提取完成，共 {len(records)} 条记录，已保存到 {output_path}")
print("样例:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

共发现 3028 个XML文件
提取完成，共 3028 条记录，已保存到 ../data/processed/dataset.json
样例:
{
  "pmc_id": "PMC524367",
  "pmid": "15462679",
  "title": "Perceived personal, social and environmental barriers to weight maintenance among young women: A community survey",
  "abstract": "Background Young women are a group at high risk of weight gain. This study examined a range of perceived personal, social and environmental barriers to physical activity and healthy eating for weight maintenance among young women, and how these varied by socioeconomic status (SES), overweight status and domestic situation. Methods In October-December 2001, a total of 445 women aged 18–32 years, selected randomly from the Australian electoral roll, completed a mailed self-report survey that included questions on 11 barriers to physical activity and 11 barriers to healthy eating (relating to personal, social and environmental factors). Height, weight and socio-demographic details were also obtained. Statistical analyses were con

## 1.检查字段完整性

In [39]:
import json

with open("../data/processed/dataset.json", encoding="utf-8") as f:
    data = json.load(f)

n = len(data)
print(f"=== 数据集总量: {n} 条 ===\n")

print("--- 字段缺失率 ---")
fields = ["pmid", "title", "abstract", "journal", "pub_date"]
missing_report = {}
for field in fields:
    missing = sum(1 for r in data if not r.get(field))
    rate = missing / n * 100
    missing_report[field] = rate
    print(f"{field:12s}: 缺失 {missing:4d} 条 / {n} ({rate:.2f}%)")

print()
abstract_missing_rate = missing_report["abstract"]
if abstract_missing_rate > 1:
    print(f"[判断] abstract 缺失率 {abstract_missing_rate:.2f}% > 1%，需要制定清洗策略。")
    print("[策略建议] abstract 是RAG检索的核心内容，缺失时无法靠填充弥补，")
    print("           建议直接丢弃这部分记录。")
else:
    print(f"[判断] abstract 缺失率 {abstract_missing_rate:.2f}% <= 1%，可直接丢弃，对数据量影响很小。")

# 解析失败（XML格式异常）导致的缺失
parse_errors = sum(1 for r in data if r.get("_parse_error"))
print(f"\n其中因XML解析报错导致整条记录缺失: {parse_errors} 条")

=== 数据集总量: 3028 条 ===

--- 字段缺失率 ---
pmid        : 缺失  275 条 / 3028 (9.08%)
title       : 缺失    0 条 / 3028 (0.00%)
abstract    : 缺失  253 条 / 3028 (8.36%)
journal     : 缺失    0 条 / 3028 (0.00%)
pub_date    : 缺失   12 条 / 3028 (0.40%)

[判断] abstract 缺失率 8.36% > 1%，需要制定清洗策略。
[策略建议] abstract 是RAG检索的核心内容，缺失时无法靠填充弥补，
           建议直接丢弃这部分记录。

其中因XML解析报错导致整条记录缺失: 0 条


## 2.基础质量分析

In [41]:
import re

valid = [r for r in data if r.get("abstract")]
print(f"有效记录数（abstract非空）: {len(valid)}\n")

# 超短摘要
short_threshold = 30 
short_abstracts = [r for r in valid if len(r["abstract"].split()) < short_threshold]
print(f"超短摘要 (<{short_threshold} 词): {len(short_abstracts)} 条 ({len(short_abstracts)/len(valid)*100:.2f}%)")
if short_abstracts:
    print(f"  示例: {short_abstracts[0]['abstract'][:150]}")
    print(f"  来源: {short_abstracts[0]['pmc_id']}")

# 编码错误检测
def has_encoding_error(text):
    return bool(re.search(r'[\x80-\x9f]|\ufffd', text))

encoding_errors = [r for r in valid if has_encoding_error(r["abstract"])]
print(f"\n疑似编码错误: {len(encoding_errors)} 条 ({len(encoding_errors)/len(valid)*100:.2f}%)")
if encoding_errors:
    print(f"  示例: {repr(encoding_errors[0]['abstract'][:100])}")
    print(f"  来源: {encoding_errors[0]['pmc_id']}")

有效记录数（abstract非空）: 2775

超短摘要 (<30 词): 200 条 (7.21%)
  示例: A  Drosophila  protein-protein interaction map was constructed using the LexA system, complementing a previous map using the GAL4 system and adding ma
  来源: PMC545799

疑似编码错误: 0 条 (0.00%)


## 3.关键字段分析

In [43]:
from collections import Counter

journals = Counter(r["journal"] for r in valid if r.get("journal"))
print(f"journal 唯一值数量: {len(journals)}")
print("出现最多的5个期刊:", journals.most_common(5))

years = Counter(r["pub_date"][:4] for r in valid if r.get("pub_date"))
print(f"\npub_date 年份跨度: {min(years)} - {max(years)}")
print("各年份文章数(部分，按年份排序前5):", dict(sorted(years.items())[:5]))

pmid_present = sum(1 for r in valid if r.get("pmid"))
print(f"\n有效记录中pmid存在的比例: {pmid_present}/{len(valid)} ({pmid_present/len(valid)*100:.2f}%)")

sample_pmid = next(r["pmid"] for r in valid if r.get("pmid"))
print(f"pmid格式示例: {sample_pmid}，是否纯数字: {sample_pmid.isdigit()}")

journal 唯一值数量: 139
出现最多的5个期刊: [('PLoS Biology', 349), ('BMC Bioinformatics', 161), ('PLoS Medicine', 90), ('BMC Cancer', 88), ('BMC Genomics', 86)]

pub_date 年份跨度: 2003 - 2024
各年份文章数(部分，按年份排序前5): {'2003': 21, '2004': 1850, '2005': 869, '2024': 23}

有效记录中pmid存在的比例: 2735/2775 (98.56%)
pmid格式示例: 15462679，是否纯数字: True


## 4.Token长度分析

In [45]:
try:
    import tiktoken
    print("tiktoken 可用")
except ImportError:
    print("tiktoken 未安装")

try:
    import transformers
    print("transformers 可用，版本:", transformers.__version__)
except ImportError:
    print("transformers 未安装")

tiktoken 未安装
transformers 可用，版本: 5.12.1


In [46]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

lengths = []
for r in valid:
    full_text = f"{r.get('title') or ''}\n{r['abstract']}"
    n_tokens = len(tokenizer.encode(full_text, truncation=False))
    lengths.append(n_tokens)

lengths.sort()
n = len(lengths)

def percentile(p):
    idx = int(n * p / 100)
    idx = min(idx, n - 1)
    return lengths[idx]

print(f"=== Token长度分布（{n}篇，title+abstract拼接，真实bge tokenizer）===\n")
for p in [5, 25, 50, 75, 90, 95, 99, 100]:
    print(f"  P{p:<3d}: {percentile(p):4d} tokens")

mean_len = sum(lengths) / n
print(f"\n  均值: {mean_len:.0f} tokens")
print(f"  最小: {lengths[0]} / 最大: {lengths[-1]}")

embedding_limit = 512
p95 = percentile(95)
p99 = percentile(99)
over_limit = sum(1 for l in lengths if l > embedding_limit)

print(f"\n=== 与bge-base-en-v1.5上限({embedding_limit} tokens)对比 ===")
print(f"  P95 = {p95} tokens")
print(f"  P99 = {p99} tokens")
print(f"  超过{embedding_limit} tokens的文章占比: {over_limit}/{n} ({over_limit/n*100:.2f}%)")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (535 > 512). Running this sequence through the model will result in indexing errors


=== Token长度分布（2775篇，title+abstract拼接，真实bge tokenizer）===

  P5  :   44 tokens
  P25 :  261 tokens
  P50 :  352 tokens
  P75 :  440 tokens
  P90 :  527 tokens
  P95 :  576 tokens
  P99 :  686 tokens
  P100:  952 tokens

  均值: 343 tokens
  最小: 8 / 最大: 952

=== 与bge-base-en-v1.5上限(512 tokens)对比 ===
  P95 = 576 tokens
  P99 = 686 tokens
  超过512 tokens的文章占比: 328/2775 (11.82%)


## 5.分层抽样

In [48]:
import random
random.seed(1)

def token_len(r):
    full_text = f"{r.get('title') or ''}\n{r['abstract']}"
    return len(tokenizer.encode(full_text, truncation=False))

short_pool = [r for r in valid if token_len(r) < 261]
medium_pool = [r for r in valid if 261 <= token_len(r) < 440]
long_pool = [r for r in valid if token_len(r) >= 440]

print(f"短文本池(<261 tokens): {len(short_pool)} 篇")
print(f"中等文本池(261-440 tokens): {len(medium_pool)} 篇")
print(f"长文本池(>=440 tokens): {len(long_pool)} 篇")

sample_short = random.sample(short_pool, 8)
sample_medium = random.sample(medium_pool, 8)
sample_long = random.sample(long_pool, 8)

print("\n=== 短文本抽样（8篇，全文）===")
for r in sample_short:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

print("\n\n=== 中等文本抽样（8篇，全文）===")
for r in sample_medium:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

print("\n\n=== 长文本抽样（8篇，全文）===")
for r in sample_long:
    print(f"\n[{r['pmc_id']}] {r['title']}")
    print(r['abstract'])
    print("-" * 80)

短文本池(<261 tokens): 693 篇
中等文本池(261-440 tokens): 1386 篇
长文本池(>=440 tokens): 696 篇

=== 短文本抽样（8篇，全文）===

[PMC549212] In vitro bioassay as a predictor of in vivo response
Background There is a substantial discrepancy between  in vitro  and  in vivo  experiments. The purpose of the present work was development of a theoretical framework to enable improved prediction of  in vivo  response from  in vitro  bioassay results. Results For dose-response curve reaches a plateau  in vitro  we demonstrated that the  in vivo  response has only one maximum. For biphasic patterns of biological response  in vitro  both the bimodal and biphasic  in vivo  responses might be observed. Conclusion As the main result of this work we have demonstrated that  in vivo  responses might be predicted from dose-effect curves measured  in vitro .
--------------------------------------------------------------------------------

[PMC387267] Calcium Dynamics of Cortical Astrocytic Networks In Vivo
Large and long-lasting 

### 高频词分析

In [50]:
from collections import Counter

STOPWORDS = set("""
the a an of and in to with for was were is are this that on at by as from
have not been also could would should may might can will shall must do does
did has had but more than two one all both other after during there most
group however such when into among each who while over only its those
the a an
""".split())

words = []
for r in valid:
    text = (r.get("title") or "") + " " + r["abstract"]
    for w in text.split():
        w = w.strip('.,();:%').lower()
        if w and w not in STOPWORDS and len(w) > 2:
            words.append(w)

freq = Counter(words)
print("=== 高频词 Top20（去停用词）===")
for w, c in freq.most_common(20):
    print(f"  {w:20s} {c}")

=== 高频词 Top20（去停用词）===
  results              2707
  patients             2125
  background           2024
  study                1860
  these                1830
  cells                1600
  methods              1526
  data                 1508
  expression           1491
  between              1404
  gene                 1387
  which                1337
  genes                1296
  analysis             1283
  cell                 1262
  using                1258
  conclusions          1174
  protein              1130
  used                 1108
  health               1087


## 6. 分块预览

In [52]:
import json
import pandas as pd

with open("../data/processed/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

chunks_df = pd.DataFrame(chunks)
print(f"总chunk数: {len(chunks_df)}")
print(f"涉及文献数: {chunks_df['doc_id'].nunique()}")

总chunk数: 3436
涉及文献数: 2775


### 样例1：智能分割（a分支）

In [108]:
multi_chunk_doc_ids = chunks_df[chunks_df["total_chunks"] > 1]["doc_id"].unique()
sample_doc_id = pd.Series(multi_chunk_doc_ids).sample(1, random_state=42).iloc[0]
sample_a = chunks_df[chunks_df["doc_id"] == sample_doc_id].sort_values("chunk_index")

print(f"\n样例1：智能分割（a分支）doc_id={sample_doc_id}，共{len(sample_a)}块")
for _, c in sample_a.iterrows():
    print(f"  chunk{c['chunk_index']} ({c['token_count']} tokens): {c['text'][:100]}...\n")



样例1：智能分割（a分支）doc_id=PMC529470，共2块
  chunk0 (394 tokens): A comparison of vas occlusion techniques: cautery more effective than ligation and excision with fas...

  chunk1 (119 tokens): Results Vasectomy with cautery was associated with a significantly more rapid progression to severe ...



### 样例2：整体不分割（b分支）

In [106]:
sample_b = chunks_df[chunks_df["total_chunks"] == 1].sample(1, random_state=42).iloc[0]
print("\n样例2：整体不分割（b分支）")
print(f"doc_id: {sample_b['doc_id']}")
print(f"token_count: {sample_b['token_count']}")
print(f"text预览: {sample_b['text'][:200]}...")


样例2：整体不分割（b分支）
doc_id: PMC545198
token_count: 43
text预览: Seasonal Patterns of Infectious Diseases. Why is that many infectious diseases, like cholera, malaria, and meningococcal meningitis, show seasonal patterns? And how can we accurately determine these p...


### 随机抽样5条

In [104]:
print(" 随机抽样5条chunk ")
for _, c in chunks_df.sample(5, random_state=1).iterrows():
    print(f"[{c['chunk_id']}] ({c['token_count']} tokens): {c['text'][:100]}...\n")


 随机抽样5条chunk 
[PMC520812_chunk1] (130 tokens): In spatial frequencies of 6 and 12 cycles per degree contrast sensitivity had decreased immediately ...

[PMC516021] (426 tokens): Human cytokine-induced killer cells have enhanced in vitro cytolytic activity via non-viral interleu...

[PMC554106] (337 tokens): What is important in evaluating health care quality? An international comparison of user views. Back...

[PMC524164] (257 tokens): Consumers as tutors – legitimate teachers?. Background The aim of this study was to research the fea...

[PMC529254_chunk0] (390 tokens): Peroxynitrite-mediated inactivation of heme oxygenases. Background Endogenous nitric oxide (NO) and ...



## 7. 质量验证

### 对生成的文本块进行质量抽样检查

In [98]:
# 1. 总块数，是否超过模型限制
print(" 1. 总块数与模型限制检查")
print(f"总chunk数: {len(chunks_df)}")

over_limit = chunks_df[chunks_df["token_count"] > 512]
print(f"超过512 tokens的chunk数: {len(over_limit)}")


 1. 总块数与模型限制检查
总chunk数: 3436
超过512 tokens的chunk数: 0


In [100]:
# 2. 文本质量

print("\n 2. 文本质量 ")

# ---- 2.1 是否包含标题 ----
import re
missing_title = chunks_df[chunks_df["source_title"].isna() | (chunks_df["source_title"].str.strip() == "")]
print(f"[标题] source_title缺失的chunk数: {len(missing_title)}")

# ---- 2.2 文本是否被不完整截断 ----
def looks_truncated_start(text):
    stripped = text.lstrip()
    if not stripped:
        return True
    return bool(re.match(r'^[.,;:)\]]', stripped)) or stripped[0].islower()

def looks_truncated_end(text):
    stripped = text.rstrip()
    if not stripped:
        return True
    return stripped[-1] not in ".!?\"'）)"

non_first_chunks = chunks_df[chunks_df["chunk_index"] > 0]
truncated_start = non_first_chunks[non_first_chunks["text"].apply(looks_truncated_start)]
print(f"[截断] 开头疑似截断的chunk数（非首块）: {len(truncated_start)} / {len(non_first_chunks)}")

not_last_chunks = chunks_df[chunks_df["chunk_index"] < chunks_df["total_chunks"] - 1]
truncated_end = not_last_chunks[not_last_chunks["text"].apply(looks_truncated_end)]
print(f"[截断] 结尾疑似截断的chunk数（非末块）: {len(truncated_end)} / {len(not_last_chunks)}")



 2. 文本质量 
[标题] source_title缺失的chunk数: 0
[截断] 开头疑似截断的chunk数（非首块）: 19 / 661
[截断] 结尾疑似截断的chunk数（非末块）: 2 / 661


In [63]:
def starts_with_stray_punct(text):
    stripped = text.lstrip()
    return bool(re.match(r'^[.,;:)\]]', stripped))

def starts_lowercase_midsentence(text):
    stripped = text.lstrip()
    return bool(stripped) and stripped[0].islower() and not starts_with_stray_punct(text)

stray_punct = non_first_chunks[non_first_chunks["text"].apply(starts_with_stray_punct)]
midsentence = non_first_chunks[non_first_chunks["text"].apply(starts_lowercase_midsentence)]

print(f"[真异常] 开头是孤立标点的chunk数: {len(stray_punct)} / {len(non_first_chunks)}")
print(f"[正常现象] 因overlap从句中开始的chunk数: {len(midsentence)} / {len(non_first_chunks)}")

[真异常] 开头是孤立标点的chunk数: 0 / 661
[正常现象] 因overlap从句中开始的chunk数: 19 / 661


### 对多块分割文献重点检查

In [71]:
# 1.总块数/超限（只看多块文献部分）
multi_chunk_docs = chunks_df[chunks_df["total_chunks"] > 1]
print(f"多块分割的文献数: {multi_chunk_docs['doc_id'].nunique()}")
multi_over_limit = multi_chunk_docs[multi_chunk_docs["token_count"] > 512]
print(f"[多块-超限] 多块文献中超过512 tokens的chunk数: {len(multi_over_limit)} / {len(multi_chunk_docs)}")


多块分割的文献数: 648
[多块-超限] 多块文献中超过512 tokens的chunk数: 0 / 1309


In [73]:
# 2 文本质量（只看多块文献部分）
multi_missing_title = multi_chunk_docs[multi_chunk_docs["source_title"].isna() | (multi_chunk_docs["source_title"].str.strip() == "")]
print(f"[多块-标题] 多块文献中标题缺失的chunk数: {len(multi_missing_title)}")

multi_non_first = multi_chunk_docs[multi_chunk_docs["chunk_index"] > 0]
multi_truncated_start = multi_non_first[multi_non_first["text"].apply(looks_truncated_start)]
print(f"[多块-截断] 多块文献中开头疑似截断的chunk数: {len(multi_truncated_start)} / {len(multi_non_first)}")

[多块-标题] 多块文献中标题缺失的chunk数: 0
[多块-截断] 多块文献中开头疑似截断的chunk数: 19 / 661


In [75]:
# 3. 结构一致性
def check_doc_consistency(group):
    n = len(group)
    expected_total = group["total_chunks"].iloc[0]
    indices = sorted(group["chunk_index"].tolist())
    return pd.Series({
        "is_continuous": indices == list(range(n)),
        "total_matches": (n == expected_total),
    })

consistency_check = multi_chunk_docs.groupby("doc_id").apply(check_doc_consistency, include_groups=False)
print(f"[多块-结构] chunk_index不连续的文献数: {(~consistency_check['is_continuous']).sum()}")
print(f"[多块-结构] total_chunks不符的文献数: {(~consistency_check['total_matches']).sum()}")

[多块-结构] chunk_index不连续的文献数: 0
[多块-结构] total_chunks不符的文献数: 0


In [77]:
# 4. 重叠部分检查
def char_overlap_length(text_a, text_b, max_check=150):
    tail = text_a[-max_check:]
    for overlap_len in range(min(len(tail), len(text_b)), 0, -1):
        if tail[-overlap_len:] == text_b[:overlap_len]:
            return overlap_len
    return 0

overlap_results = []
for doc_id, group in multi_chunk_docs.groupby("doc_id"):
    group_sorted = group.sort_values("chunk_index").reset_index(drop=True)
    for i in range(len(group_sorted) - 1):
        overlap_chars = char_overlap_length(group_sorted.loc[i, "text"], group_sorted.loc[i + 1, "text"])
        overlap_results.append({"doc_id": doc_id, "chunk_pair": f"{i}->{i+1}", "overlap_chars": overlap_chars})

overlap_df = pd.DataFrame(overlap_results)
print(f"[多块-重叠] 共检查 {len(overlap_df)} 对相邻chunk")
print(f"[多块-重叠] 零重叠的chunk对数: {(overlap_df['overlap_chars'] == 0).sum()}")
print(f"[多块-重叠] 重叠字符数分布:\n{overlap_df['overlap_chars'].describe()}")

[多块-重叠] 共检查 661 对相邻chunk
[多块-重叠] 零重叠的chunk对数: 568
[多块-重叠] 重叠字符数分布:
count    661.000000
mean      16.900151
std       43.090029
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      150.000000
Name: overlap_chars, dtype: float64


In [102]:
zero_pairs = overlap_df[overlap_df["overlap_chars"] == 0].sample(3, random_state=1)

for _, row in zero_pairs.iterrows():
    doc_id = row["doc_id"]
    i0, i1 = row["chunk_pair"].split("->")
    group = multi_chunk_docs[multi_chunk_docs["doc_id"] == doc_id].sort_values("chunk_index").reset_index(drop=True)
    text_a = group.loc[int(i0), "text"]
    text_b = group.loc[int(i1), "text"]
    print(f"doc_id={doc_id}, {row['chunk_pair']} ")
    print(f"chunk{i0}结尾: ...{text_a[-100:]}")
    print(f"chunk{i1}开头: {text_b[:100]}...")
    print()

doc_id=PMC549549, 0->1 
chunk0结尾: ...omid) assay, and cells at lower density in culture were found to be more sensitive to GSH depletion.
chunk1开头: Cell viability was tested by MTT (3-[4,5-dimethylthiazol-2-yl]-2,5-diphenyl tetrazolium bromid) assa...

doc_id=PMC516168, 0->1 
chunk0结尾: ...ET between subunits fused to variants of green fluorescence protein (CFP, YFP) in transfected cells.
chunk1开头: Conclusions All three sets of results show that V-ATPase contains VHA-a protein that interacts in a ...

doc_id=PMC520811, 0->1 
chunk0结尾: ... continuous variables were compared using the Student t test or Mann-Whitney U test, as appropriate.
chunk1开头: For intergroup comparisons, categorical variables were compared using the chi-squared test and conti...



In [81]:
# 抽查几个零重叠样例，看看被跳过的句子有多长
zero_overlap_sample = overlap_df[overlap_df["overlap_chars"] == 0].sample(5, random_state=2)

for _, row in zero_overlap_sample.iterrows():
    doc_id = row["doc_id"]
    i0, i1 = row["chunk_pair"].split("->")
    group = multi_chunk_docs[multi_chunk_docs["doc_id"] == doc_id].sort_values("chunk_index").reset_index(drop=True)
    chunk0_tokens = group.loc[int(i0), "token_count"]
    chunk1_tokens = group.loc[int(i1), "token_count"]
    print(f"doc_id={doc_id}, chunk{i0}({chunk0_tokens} tokens) -> chunk{i1}({chunk1_tokens} tokens)")

doc_id=PMC555548, chunk0(391 tokens) -> chunk1(174 tokens)
doc_id=PMC538285, chunk0(401 tokens) -> chunk1(206 tokens)
doc_id=PMC549207, chunk0(353 tokens) -> chunk1(223 tokens)
doc_id=PMC517721, chunk0(405 tokens) -> chunk1(315 tokens)
doc_id=PMC544575, chunk0(407 tokens) -> chunk1(114 tokens)


In [91]:
# 假设一：末句token数超出80-token重叠预算，分割器放弃重叠

def count_str_tokens(text):
    return len(tokenizer.encode(text, truncation=False))

def get_last_sentence_tokens(text):
    for sep in ["\n\n", "\n", ". "]:
        if sep in text:
            last_piece = text.split(sep)[-1]
            return count_str_tokens(last_piece)
    return count_str_tokens(text)

zero_overlap_check = overlap_df[overlap_df["overlap_chars"] == 0].sample(10, random_state=3)
for _, row in zero_overlap_check.iterrows():
    doc_id = row["doc_id"]
    i0, i1 = row["chunk_pair"].split("->")
    group = multi_chunk_docs[multi_chunk_docs["doc_id"] == doc_id].sort_values("chunk_index").reset_index(drop=True)
    text_a = group.loc[int(i0), "text"]
    last_sent_tokens = get_last_sentence_tokens(text_a)
    print(f"doc_id={doc_id}: 末句token数≈{last_sent_tokens}")

doc_id=PMC535559: 末句token数≈31
doc_id=PMC549549: 末句token数≈69
doc_id=PMC539271: 末句token数≈36
doc_id=PMC551604: 末句token数≈24
doc_id=PMC544563: 末句token数≈38
doc_id=PMC544847: 末句token数≈24
doc_id=PMC514528: 末句token数≈83
doc_id=PMC538287: 末句token数≈56
doc_id=PMC521080: 末句token数≈35
doc_id=PMC549187: 末句token数≈27


In [96]:
# 检测方法过于严格（要求逐字符精确匹配）导致误判 

def sentence_reappears(text_a, text_b, check_chars=200):
    # 取chunk0最后一句（去除首尾空白，标准化多余空格）
    for sep in ["\n\n", "\n", ". "]:
        if sep in text_a:
            last_sent = text_a.split(sep)[-1].strip()
            break
    else:
        last_sent = text_a.strip()

    last_sent_norm = " ".join(last_sent.split())  # 标准化空白
    text_b_norm = " ".join(text_b[:check_chars].split())

    return last_sent_norm in text_b_norm, last_sent_norm[:50]

zero_overlap_check2 = overlap_df[overlap_df["overlap_chars"] == 0].sample(10, random_state=3)
for _, row in zero_overlap_check2.iterrows():
    doc_id = row["doc_id"]
    i0, i1 = row["chunk_pair"].split("->")
    group = multi_chunk_docs[multi_chunk_docs["doc_id"] == doc_id].sort_values("chunk_index").reset_index(drop=True)
    text_a = group.loc[int(i0), "text"]
    text_b = group.loc[int(i1), "text"]
    found, preview = sentence_reappears(text_a, text_b)
    print(f"doc_id={doc_id}: 末句是否重新出现在chunk1开头附近: {found} | 末句预览: {preview}...")

doc_id=PMC535559: 末句是否重新出现在chunk1开头附近: False | 末句预览: 7% (6/86) of tumours exhibited considerable inhibi...
doc_id=PMC549549: 末句是否重新出现在chunk1开头附近: True | 末句预览: Cell viability was tested by MTT (3-[4,5-dimethylt...
doc_id=PMC539271: 末句是否重新出现在chunk1开头附近: False | 末句预览: In this experiment when the tumor volumes reached ...
doc_id=PMC551604: 末句是否重新出现在chunk1开头附近: False | 末句预览: Our analyses provide a platform for experimental d...
doc_id=PMC544563: 末句是否重新出现在chunk1开头附近: False | 末句预览: In multivariate regression models, household size,...
doc_id=PMC544847: 末句是否重新出现在chunk1开头附近: False | 末句预览: Contrary to anticipated results, high enzymatic ac...
doc_id=PMC514528: 末句是否重新出现在chunk1开头附近: False | 末句预览: These included antigen chaperone molecules (HSP-70...
doc_id=PMC538287: 末句是否重新出现在chunk1开头附近: False | 末句预览: The significant stress reductions in both the fish...
doc_id=PMC521080: 末句是否重新出现在chunk1开头附近: False | 末句预览: Importantly, concentrations of THC that inhibit vi...
doc_id=PMC549187: 末句是否重新出现在chunk1开头附近: